# Colab smoke test

Checks that the project installs and runs on a fresh Colab runtime, before the real notebook is built:

1. clones the repo and installs the pinned requirements on top of Colab's own PyTorch;
2. prints the Python, PyTorch and CUDA versions and the GPU;
3. loads both encoders (DINOv2-small and V-JEPA 2 ViT-L) and embeds one frame;
4. optional: renders MuJoCo headless and replays a short push with the Panda.

Use a GPU runtime (**Runtime > Change runtime type > T4 GPU**), then **Runtime > Run all**.
The last cell prints a summary to copy back.

In [ ]:
# ---- Which repo to test. EDIT HERE if the repo lives elsewhere. ----
REPO_URL = "https://github.com/well-iam/humanoid-intern-challenge"
BRANCH = "pushing-world-model"  # "main" once the work is merged
# ---------------------------------------------------------------------

import os
import subprocess
import sys

REPO = "/content/humanoid-intern-challenge"
DATA = "/content/pwm-data"
summary = []

if not os.path.exists(REPO):
    url, token = REPO_URL, None
    try:  # only while the repo is private: a GitHub token saved as the Colab secret GH_TOKEN (key icon, left)
        from google.colab import userdata

        token = userdata.get("GH_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO], capture_output=True, text=True)
    if clone.returncode != 0:
        raise SystemExit("git clone failed:\n" + clone.stderr.replace(token or "\0", "<token>"))
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
commit = subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
summary.append(f"repo: {REPO_URL} @ {BRANCH}, {commit}")
print(summary[-1])

In [ ]:
# Pinned requirements (exported from uv.lock, without PyTorch: Colab's own PyTorch is used).
!pip install -q -r {REPO}/requirements.txt
sys.path.insert(0, f"{REPO}/src")

# A package that was already imported when pip replaced it keeps its old version until a restart.
import importlib.metadata as md

stale = []
for module, dists in md.packages_distributions().items():
    loaded = getattr(sys.modules.get(module), "__version__", None)
    try:
        installed = md.version(dists[0])
    except md.PackageNotFoundError:
        continue
    if isinstance(loaded, str) and loaded != installed:
        stale.append(f"{module} {loaded} -> {installed}")
if stale:
    print("RESTART NEEDED: Runtime > Restart session, then Runtime > Run all again. Changed:", ", ".join(stale))
else:
    print("install OK, no restart needed")
summary.append(f"restart needed: {', '.join(stale) or 'no'}")

In [ ]:
# The environment: write this down as the tested runtime.
import re

import torch

pinned_torch = re.search(r'"torch==([^"]+)"', open(f"{REPO}/pyproject.toml").read())[1]
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)"
lines = [
    f"Python {sys.version.split()[0]}",
    f"Colab runtime: {os.environ.get('COLAB_RELEASE_TAG', 'unknown')}",
    f"PyTorch {torch.__version__} (local pin {pinned_torch}), CUDA {torch.version.cuda}, cuDNN {torch.backends.cudnn.version()}",
    f"GPU: {gpu}",
]
for package in ["numpy", "transformers", "huggingface-hub", "zarr", "mujoco", "mink", "gym-pusht", "pymunk", "opencv-python"]:
    try:
        lines.append(f"{package} {md.version(package)}")
    except md.PackageNotFoundError:
        lines.append(f"{package} MISSING")
for module in ["zarr", "mujoco", "mink", "cv2", "pymunk", "gym_pusht"]:
    try:
        imported = __import__(module)
        lines.append(f"import {module}: OK {getattr(imported, '__version__', '')}")
    except Exception as error:
        lines.append(f"import {module}: FAILED {type(error).__name__}: {error}")
print("\n".join(lines))
summary += lines

In [ ]:
# Both encoders, through the encoder adapters, on one frame of the toy world.
import time
import traceback

import numpy as np

from pwm.datasets import toy
from pwm.encoders import get_encoder

frame = toy.make_episodes(1, 0)[0].frames[:1]  # (1, 64, 64, 3) uint8
for name, dims in [("dinov2", 384), ("vjepa2", 1024)]:
    try:
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        encoder = get_encoder(name)
        loaded = time.perf_counter() - started
        started = time.perf_counter()
        embedding = encoder.encode(frame)
        encoded = time.perf_counter() - started
        assert embedding.shape == (1, 256, dims), embedding.shape
        assert np.isfinite(embedding).all(), "NaN or inf in the embedding"
        peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
        line = (f"{name}: OK on {encoder.device}, embedding {embedding.shape}, std {embedding.std():.3f}, "
                f"load {loaded:.0f} s, encode {encoded:.2f} s, peak GPU {peak:.2f} GB")
        del encoder
        torch.cuda.empty_cache()
    except Exception as error:
        traceback.print_exc()
        line = f"{name}: FAILED {type(error).__name__}: {error}"
    print(line)
    summary.append(line)

### Optional: MuJoCo rendering

Headless rendering for the Panda replay. A failure here does not affect the checks above.
It tries EGL (the GPU) first, then OSMesa (the CPU) if EGL fails.

In [ ]:
from IPython.display import Image, display

for backend in ["egl", "osmesa"]:
    if backend == "osmesa":
        subprocess.run("apt-get -qq install -y libosmesa6-dev > /dev/null", shell=True)
    env = {**os.environ, "MUJOCO_GL": backend, "PYTHONPATH": f"{REPO}/src"}
    run = subprocess.run([sys.executable, f"{REPO}/scripts/render_check.py", "--root", DATA],
                         env=env, capture_output=True, text=True)
    result = "PASS" if run.returncode == 0 else "FAIL"
    print(f"--- MUJOCO_GL={backend}: {result}\n{run.stdout}{run.stderr[-3000:]}")
    summary.append(f"render, MUJOCO_GL={backend}: {result}")
    if run.returncode == 0:
        display(Image(f"{DATA}/render_check.png"))
        break

Copy everything below and paste it back.

In [ ]:
print("\n".join(summary))